# REWIND M2 Control800 / Improved800
T4 런타임에서 실행한다. WORK.md의 비공개 파일을 준비하고 경로를 지정한다. 결과는 Drive에 보존한다.

In [ ]:
import os, subprocess, sys
from pathlib import Path
from google.colab import drive
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(), 'Tesla T4 required'
drive.mount('/content/drive')
repo = Path('/content/rewind_bone_control')
revision = 'e3e884643a75c876ae8514a04ea2a919bc70b984'
if not repo.exists():
    subprocess.run(['git','clone','--branch','experiment/m2-bone-length-control-20261119','https://github.com/darkkyoung/REWIND-like-prototype.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run(['git','checkout','--detach',revision],check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()==revision
assert not subprocess.check_output(['git','status','--porcelain']).strip()
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],check=True)

In [ ]:
base = Path('/content/drive/MyDrive/rewind_bone')
inputs, parent, poses = base/'inputs', base/'parent', base/'poses'
candidate, results = base/'candidate', base/'results'
package = 'experiments.final_m2.bone_length_loss_20261117'
run('-m','unittest',package+'.test_bone','-v')
run('-m','unittest','discover','-s','tests','-p','test_m2.py','-v')

In [ ]:
run('-m',package+'.prepare_candidate',poses,candidate)

In [ ]:
run('-m',package+'.run_experiment','--input-dir',inputs,'--checkpoint-dir',parent,'--unused-motion',candidate/'joyful57.npz','--output',results)
print((results/'evaluation.json').read_text())